# Pipeline Ingesta Incremental

## Descripción General
Este notebook orquesta la ingesta incremental de archivos Excel en tablas Delta Lake.
Soporta tanto cargas completas como actualizaciones incrementales con seguimiento de watermark.

## Prerrequisitos
- Acceso a la capa Bronze volumes (`{environment}_bronze_prelanding_minsur`)
- Archivos Excel almacenados en la ruta del servidor de archivos configurada
- Existencia de la tabla Delta para cargas incrementales (si se configura watermark)

## Sección 1: Importación de Librerías y Configuración del Entorno

**Propósito**: Importar las librerías necesarias y configurar los ajustes de Spark/Pandas para un rendimiento
y compatibilidad óptimos.

**Configuraciones Clave**:
- Deshabilitar la optimización Arrow para prevenir problemas de serialización entre Spark y Pandas
- Suprimir advertencias de formato de openpyxl (no críticas)
- Importar utilidades personalizadas desde el notebook `Ingesta Funciones`

In [0]:
%run "../Funciones/Ingesta Funciones"

In [0]:
# Importar librerías de procesamiento de datos
import warnings
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructField
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import col, lit, max as spark_max
from delta.tables import DeltaTable

# CRÍTICO: Deshabilitar Arrow para evitar problemas de compatibilidad en la serialización Spark-Pandas
# Arrow puede causar errores de "Trabajador de Python salió inesperadamente" con tipos complejos
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")
spark.conf.set("spark.sql.ansi.enabled", "false")

# Suprimir advertencias no críticas de openpyxl sobre el formato de Excel
# Estas advertencias no afectan la lectura de datos pero saturan los registros
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

In [0]:
# Constantes para validación y procesamiento
VALID_DATETIME_FORMAT = "%Y-%m-%d %H:%M:%S"
SUPPORTED_EXCEL_EXTENSIONS = ['.xlsx', '.xls', '.xlsm']
DEFAULT_HEADER_ROWS = 1

print("Entorno configurado exitosamente")
print(f"   Optimización Arrow: DESHABILITADA")
print(f"   Formato de fecha/hora predeterminado: {VALID_DATETIME_FORMAT}")

## Sección 2: Definición de Parámetros y Configuración de Widgets

**Propósito**: Definir parámetros de ejecución utilizando los widgets de Databricks para una ejecución
flexible y basada en configuración sin cambios en el código.

### Categorías de Widgets:
1. **Identificación Principal**: Id del Registro, Entorno
2. **Configuración de Origen**: Ruta de Excel, nombre de hoja, rango de datos
3. **Configuración de Esquema**: Mapeo de columnas, orientación, manejo de encabezados
4. **Enriquecimiento de Metadatos**: Fecha del reporte, extracción de nombre de pronóstico
5. **Lógica Incremental**: Columnas de watermark, período de retroceso, columnas ID

In [0]:
# Parámetros principales de identificación
dbutils.widgets.text("Id", "", "Id")
dbutils.widgets.text("Environment", "", "Entorno")

# Configuración del archivo fuente
dbutils.widgets.text("ExcelPath", "", "Ruta de Excel")
dbutils.widgets.text("TableName", "", "Nombre de Tabla")
dbutils.widgets.text("SheetName", "", "Nombre de Hoja")
dbutils.widgets.text("DataRange", "", "Rango de Datos")

# Configuración de esquema
dbutils.widgets.text("IndexColumns", "", "Columnas Índice")
dbutils.widgets.text("CustomColumnNames", "", "Nombres de Columnas Personalizados")
dbutils.widgets.text("ColumnsToSkip", "", "Columnas a Omitir")
dbutils.widgets.text("RowsToSkip", "", "Filas a Omitir")
dbutils.widgets.text("IsHorizontal", "", "Es Horizontal")
dbutils.widgets.text("MergeNHeaders", "", "Combinar N Encabezados")

# Parámetros de enriquecimiento de metadatos
dbutils.widgets.text("ReportDate", "", "Fecha de Reporte")
dbutils.widgets.text("ForecastName", "", "Nombre de Pronóstico")

# Configuración de carga incremental
dbutils.widgets.text("IdColumns", "", "Columnas ID")
dbutils.widgets.text("WatermarkColumn", "", "Columna Watermark")
dbutils.widgets.text("WatermarkLookBack", "", "Retroceso de Watermark")
dbutils.widgets.text("WatermarkVal", "", "Valor de Watermark")
dbutils.widgets.text("Datetime", "", "Fecha/Hora")

print("Todos los widgets definidos exitosamente")
print(f"   Total de parámetros: 16")

## Sección 3: Obtención y Validación de Parámetros

**Propósito**: Obtener los valores de los widgets, realizar validación de entrada y construir
rutas de archivo dinámicas con reemplazo de tokens de fecha/hora.

**Lógica de Validación**:
- Campos obligatorios: Id, Entorno, RutaExcel, NombreTabla, NombreHoja
- Análisis de booleanos con valor predeterminado Falso
- Análisis de numéricos con seguridad de tipos
- Reemplazo de tokens de fecha/hora para rutas dinámicas

In [0]:
# ==============================================================================
# SECCIÓN 3.1: Obtener Valores de los Widgets
# ==============================================================================

# Obtener parámetros principales
id_record = dbutils.widgets.get("Id")
environment = dbutils.widgets.get("Environment")
excel_file_path = dbutils.widgets.get("ExcelPath")
table_name = dbutils.widgets.get("TableName")
excel_sheet_name = dbutils.widgets.get("SheetName")
excel_cell_range = dbutils.widgets.get("DataRange")

# Obtener configuración de esquema
index_columns = dbutils.widgets.get("IndexColumns")
custom_column_names = dbutils.widgets.get("CustomColumnNames")
skip_col_range = dbutils.widgets.get("ColumnsToSkip")
skip_row_range = dbutils.widgets.get("RowsToSkip")

# Analizar parámetro booleano con validación explícita
is_horizontal_raw = dbutils.widgets.get("IsHorizontal")
is_horizontal = str(is_horizontal_raw).lower().strip() == "true"

# Analizar parámetro numérico con manejo de errores
try:
    header_rows = int(dbutils.widgets.get("MergeNHeaders") or DEFAULT_HEADER_ROWS)
    if header_rows < 1:
        header_rows = DEFAULT_HEADER_ROWS
        print(f"Valor de filas de encabezado inválido, usando valor predeterminado: {DEFAULT_HEADER_ROWS}")
except ValueError:
    header_rows = DEFAULT_HEADER_ROWS
    print(f"Valor de filas de encabezado no numérico, usando valor predeterminado: {DEFAULT_HEADER_ROWS}")

# Obtener parámetros de metadatos
report_date = dbutils.widgets.get("ReportDate")
forecast_name = dbutils.widgets.get("ForecastName")

# Obtener parámetros incrementales
id_columns = dbutils.widgets.get("IdColumns")
watermark_column = dbutils.widgets.get("WatermarkColumn")
watermark_look_back = dbutils.widgets.get("WatermarkLookBack")
watermark_val = dbutils.widgets.get("WatermarkVal")

# ==============================================================================
# SECCIÓN 3.2: Manejar Fecha/Hora de Ejecución
# ==============================================================================

# Obtener fecha/hora actual del widget o generar una nueva
datetime_widget = dbutils.widgets.get("Datetime")
if datetime_widget and datetime_widget.strip():
    try:
        # Validar el formato de fecha/hora proporcionado
        current_datetime_obj = datetime.strptime(datetime_widget.strip(), VALID_DATETIME_FORMAT)
        current_datetime = datetime_widget.strip()
    except ValueError:
        # Revertir a la hora UTC actual si el formato es inválido
        print(f"Formato de fecha/hora '{datetime_widget}' inválido, usando la hora UTC actual")
        from datetime import timezone
        current_datetime_obj = datetime.now(timezone.utc)
        current_datetime = current_datetime_obj.strftime(VALID_DATETIME_FORMAT)
else:
    # Valor predeterminado a la hora UTC actual
    from datetime import timezone
    current_datetime_obj = datetime.now(timezone.utc)
    current_datetime = current_datetime_obj.strftime(VALID_DATETIME_FORMAT)

# ==============================================================================
# SECCIÓN 3.3: Validar Parámetros Obligatorios
# ==============================================================================

validation_errors = []

if not id_record or not id_record.strip():
    validation_errors.append("'Id' es obligatorio")
if not environment or not environment.strip():
    validation_errors.append("'Environment' es obligatorio")
if not excel_file_path or not excel_file_path.strip():
    validation_errors.append("'ExcelPath' es obligatorio")
if not table_name or not table_name.strip():
    validation_errors.append("'TableName' es obligatorio")
if not excel_sheet_name or not excel_sheet_name.strip():
    validation_errors.append("'SheetName' es obligatorio")

if validation_errors:
    error_msg = "FALLO EN LA VALIDACIÓN DE PARÁMETROS:\\n" + "\\n".join(validation_errors)
    raise ValueError(error_msg)

print("Parámetros obligatorios validados")

# ==============================================================================
# SECCIÓN 3.4: Construir Rutas de Archivo con Tokens de Fecha/Hora
# ==============================================================================

# Aplicar reemplazo de tokens de fecha/hora a la ruta del archivo
# Ejemplo: "data/archivo_{yyyyMMddHH}.xlsx" -> "data/archivo_2026020509.xlsx"
try:
    path_with_tokens = excel_file_path.replace(".xlsx", "_{yyyyMMddHH}.xlsx")
    final_path = apply_datetime_tokens(path_with_tokens, current_datetime_obj)
except Exception as e:
    raise RuntimeError(f"Error al aplicar tokens de fecha/hora a la ruta '{excel_file_path}': {str(e)}")

# Construir ruta completa del volumen DBFS
# Formato: /Volumes/{environment}_bronze_prelanding_minsur/marcobre_fileserver/marcobre_fileserver/{ruta}
volume_base_path = f"/Volumes/{environment}_bronze_prelanding_minsur"
input_excel_path = f"{volume_base_path}/marcobre_fileserver/marcobre_fileserver/{final_path}"

# Validar extensión del archivo
file_ext = Path(final_path).suffix.lower()
if file_ext not in SUPPORTED_EXCEL_EXTENSIONS:
    raise ValueError(f"Extensión de archivo '{file_ext}' no soportada. Soportadas: {SUPPORTED_EXCEL_EXTENSIONS}")

# ==============================================================================
# SECCIÓN 3.5: Registrar Resumen de Configuración
# ==============================================================================

print("Configuración de Ejecución:")
print(f"   ID de Registro:   {id_record}")
print(f"   Entorno:          {environment}")
print(f"   Ruta de Entrada:  {input_excel_path}")
print(f"   Tabla Destino:    {table_name}")
print(f"   Nombre de Hoja:   {excel_sheet_name}")
print(f"   Orientación:      {'Horizontal' if is_horizontal else 'Vertical'}")
print(f"   Filas de Encabezado: {header_rows}")
print(f"   Hora de Ejecución: {current_datetime}")

## Sección 4: Análisis de Rango de Excel y Preparación de Parámetros

**Propósito**: Analizar las configuraciones de rango de Excel para determinar las columnas y filas exactas
a extraer, manejando configuraciones de omisión para ambas dimensiones.

**Lógica**:
- Analizar el rango de celdas (ej., "A1:Z100")
- Aplicar rangos de omisión de columnas
- Aplicar rangos de omisión de filas
- Generar parámetros listos para pandas (usecols, skiprows, nrows)

In [0]:
# ==============================================================================
# SECCIÓN 4.1: Analizar Parámetros de Rango de Excel
# ==============================================================================

try:
    params = parse_excel_range_advanced(
        table_range=excel_cell_range,
        skip_col_range=skip_col_range,
        skip_row_range=skip_row_range
    )
    
    # Validar parámetros analizados
    if not params.get('usecols'):
        raise ValueError("No se seleccionaron columnas para la extracción")
        
    print("Rango de Excel analizado exitosamente")
    print(f"   Usar columnas:   {params.get('usecols')}")
    print(f"   Omitir filas:     {params.get('skiprows')}")
    print(f"   Máx filas:        {params.get('nrows')}")
    
except Exception as e:
    raise RuntimeError(f"Error al analizar el rango de Excel '{excel_cell_range}': {str(e)}")

## Sección 5: Extracción de Datos de Excel

**Propósito**: Leer datos sin procesar de Excel usando pandas con un manejo robusto de errores.
Soporta orientaciones tanto horizontales (transpuestas) como verticales.

**Proceso**:
1. Leer Excel sin encabezados (procesamiento manual de encabezados)
2. Manejar orientación horizontal (transponer si es necesario)
3. Aplicar mapeo de columnas personalizado o construir encabezados desde las primeras N filas
4. Validar el éxito de la extracción de datos

In [0]:
# Leer datos sin procesar de Excel sin encabezados (procesamiento manual de encabezados)
df = pd.read_excel(
    input_excel_path,
    sheet_name=excel_sheet_name,
    usecols=params['usecols'],
    skiprows=params['skiprows'],
    nrows=params['nrows'],
    header=None,  # Los encabezados se procesarán manualmente según la configuración
    engine="openpyxl"
)

# Manejar orientación horizontal (datos transpuestos)
if is_horizontal:
    # Transponer el DataFrame y reconstruir la estructura
    df_t = df.T.reset_index(drop=True)
    
    # Aplicar mapeo de columnas personalizado si se especifica
    if index_columns != "" and custom_column_names != "":
        col_indexes = [int(i) for i in index_columns.split(",")]
        new_column_names = [c.strip() for c in custom_column_names.split(",")]
        
        # Validar consistencia del mapeo de columnas
        if len(col_indexes) != len(new_column_names):
            raise ValueError(
                f"El número de columnas índice ({len(col_indexes)}) no coincide con "
                f"el número de nombres de columna personalizados ({len(new_column_names)})"
            )
        
        # Seleccionar columnas especificadas y aplicar nombres personalizados
        df_final = df_t.iloc[:, col_indexes]
        df_final.columns = new_column_names
    else:
        # Construir encabezados desde las primeras N filas y eliminarlas de los datos
        headers = build_headers(df_t, header_rows)
        df_t.columns = headers
        df_final = df_t.iloc[header_rows:].reset_index(drop=True)
else:
    # Procesamiento de orientación vertical
    if index_columns != "" and custom_column_names != "":
        # Selección de columnas personalizada para orientación vertical
        col_indexes = [int(i) - 1 for i in index_columns.split(",")]
        new_column_names = [c.strip() for c in custom_column_names.split(",")]
        
        # Validar mapeo
        if len(col_indexes) != len(new_column_names):
            raise ValueError(
                f"El número de columnas índice ({len(col_indexes)}) no coincide con "
                f"el número de nombres de columna personalizados ({len(new_column_names)})"
            )
        
        df_final = df.iloc[:, col_indexes]
        df_final.columns = new_column_names
    else:
        # Construcción automática de encabezados desde las primeras N filas
        headers = build_headers(df, header_rows)
        df.columns = headers
        # Eliminar filas de encabezado de los datos reales
        df = df.iloc[header_rows:].reset_index(drop=True)
        df_final = df.copy()

## Sección 6: Enriquecimiento de Metadatos

**Propósito**: Extraer metadatos adicionales de celdas específicas de Excel y agregarlos
como columnas para trazabilidad y auditoría de datos.

**Campos de Metadatos**:
- `report_date`: Extraído de la celda configurada (ej., B2)
- `forecast`: Extraído de la celda configurada (ej., C2)

**Seguridad**: Usa `.copy()` para evitar la advertencia SettingWithCopyWarning

In [0]:
# ==============================================================================
# SECCIÓN 6.1: Inicializar Variables de Metadatos
# ==============================================================================

report_date_val = None
forecast_name_val = None

# ==============================================================================
# SECCIÓN 6.2: Extraer Fecha de Reporte
# ==============================================================================

if report_date and report_date.strip():
    try:
        print(f"Extrayendo fecha de reporte de la celda: {report_date.strip()}")
        
        # Analizar rango de celda única
        report_params = parse_excel_range_advanced(
            f"{report_date.strip()}:{report_date.strip()}"
        )
        
        # Leer la celda específica
        report_df = pd.read_excel(
            input_excel_path,
            sheet_name=excel_sheet_name,
            usecols=report_params['usecols'],
            skiprows=report_params['skiprows'],
            nrows=report_params['nrows'],
            header=None,
            engine="openpyxl"
        )
        
        # Extraer valor y agregar al dataframe
        if not report_df.empty:
            report_date_val = str(report_df.iloc[0, 0])
            # Usar .copy() para evitar SettingWithCopyWarning
            df_final = df_final.copy()
            df_final["report_date"] = report_date_val
            print(f"   Valor de fecha de reporte: {report_date_val}")
        else:
            print("   No se encontró valor de fecha de reporte")
            
    except Exception as e:
        print(f"   Error al extraer fecha de reporte: {str(e)}")

# ==============================================================================
# SECCIÓN 6.3: Extraer Nombre de Pronóstico
# ==============================================================================

if forecast_name and forecast_name.strip():
    try:
        print(f"Extrayendo nombre de pronóstico de la celda: {forecast_name.strip()}")
        
        # Analizar rango de celda única
        forecast_params = parse_excel_range_advanced(
            f"{forecast_name.strip()}:{forecast_name.strip()}"
        )
        
        # Leer la celda específica
        forecast_df = pd.read_excel(
            input_excel_path,
            sheet_name=excel_sheet_name,
            usecols=forecast_params['usecols'],
            skiprows=forecast_params['skiprows'],
            nrows=forecast_params['nrows'],
            header=None,
            engine="openpyxl"
        )
        
        # Extraer valor y agregar al dataframe
        if not forecast_df.empty:
            forecast_name_val = str(forecast_df.iloc[0, 0])
            # Usar .copy() para evitar SettingWithCopyWarning
            df_final = df_final.copy()
            df_final["forecast"] = forecast_name_val
            print(f"   Valor de nombre de pronóstico: {forecast_name_val}")
        else:
            print("   No se encontró valor de nombre de pronóstico")
            
    except Exception as e:
        print(f"   Error al extraer nombre de pronóstico: {str(e)}")

# ==============================================================================
# SECCIÓN 6.4: Registrar Resultados de Enriquecimiento
# ==============================================================================

print(f"Enriquecimiento de metadatos completado")
if report_date_val:
    print(f"   Columna agregada 'report_date': {report_date_val}")
if forecast_name_val:
    print(f"   Columna agregada 'forecast': {forecast_name_val}")

## Sección 7: Limpieza y Estandarización de Datos

**Propósito**: Aplicar transformaciones integrales de calidad de datos para garantizar
datos consistentes y limpios adecuados para el almacenamiento en Delta Lake.

**Transformaciones**:
1. **Estandarización de Columnas**: Eliminar caracteres especiales, normalizar a nombres compatibles con SQL
2. **Eliminación de Duplicados**: Resolver nombres de columna duplicados con sufijos numéricos
3. **Normalización de Tipos**: Convertir cadenas a numérico cuando sea posible, estandarizar fechas
4. **Limpieza de Filas**: Eliminar filas completamente vacías (todos NaN)

In [0]:
# ==============================================================================
# SECCIÓN 7.1: Almacenar Columnas Originales para Auditoría
# ==============================================================================

original_columns = df_final.columns.tolist()

# ==============================================================================
# SECCIÓN 7.2: Estandarizar Nombres de Columna
# ==============================================================================

try:
    # Aplicar transformaciones de nombres de columna compatibles con SQL
    df_final.columns = [sanitize_column(c) for c in df_final.columns]
    print(f"Nombres de columna estandarizados")
except Exception as e:
    raise RuntimeError(f"Error en la estandarización de columnas: {str(e)}")

# ==============================================================================
# SECCIÓN 7.3: Resolver Nombres de Columna Duplicados
# ==============================================================================

try:
    # Corregir duplicados agregando sufijos numéricos (ej., nombre, nombre_2, nombre_3)
    df_final.columns = fix_duplicate_columns(df_final.columns)
    duplicate_count = len(df_final.columns) - len(set(df_final.columns))
    if duplicate_count == 0:
        print("No se encontraron nombres de columna duplicados")
    else:
        print(f"Se resolvieron {duplicate_count} nombre(s) de columna duplicados")
except Exception as e:
    raise RuntimeError(f"Error en la resolución de duplicados de columna: {str(e)}")

# ==============================================================================
# SECCIÓN 7.4: Normalizar Tipos de Datos
# ==============================================================================

try:
    # Convertir cadenas a numérico cuando sea posible, estandarizar formatos de fecha
    df_final = normalize_dataframe_types(df_final)
    print("Tipos de datos normalizados")
except Exception as e:
    print(f"Advertencia en la normalización de tipos: {str(e)}")

# ==============================================================================
# SECCIÓN 7.5: Eliminar Filas Vacías
# ==============================================================================

try:
    row_count_before = len(df_final)
    df_final = drop_all_nan_rows(df_final)
    rows_removed = row_count_before - len(df_final)
    print(f"Se eliminaron {rows_removed} fila(s) vacía(s)")
except Exception as e:
    print(f"Advertencia en la limpieza de filas: {str(e)}")

# ==============================================================================
# SECCIÓN 7.6: Construir Sufijo de Pronóstico para el Nombre de la Tabla
# ==============================================================================

final_forecast_name = ""
if forecast_name_val and forecast_name_val.strip():
    # Normalizar: minúsculas, reemplazar espacios con guiones bajos
    final_forecast_name = f"_{forecast_name_val.lower().replace(' ', '_')}"

# ==============================================================================
# SECCIÓN 7.7: Mostrar Registro de Auditoría de Transformación de Columnas
# ==============================================================================

print(f"\nRegistro de Auditoría de Transformación de Columnas:")
print(f"   {'Original':<30} -> {'Estandarizado':<30}")
print(f"   {'-'*30}    {'-'*30}")
for original, sanitized in zip(original_columns, df_final.columns):
        print(f"   {original:<30} -> {sanitized:<30}")

print(f"\nEsquema final: {len(df_final.columns)} columnas, {len(df_final)} filas")

## Sección 8: Conversión a Spark y Procesamiento de Watermark

**Propósito**: Convertir DataFrame de pandas a DataFrame de Spark, calcular valores de watermark
y aplicar filtrado de retroceso para cargas incrementales.

**Operaciones Clave**:
- Convertir a DataFrame de Spark para operaciones Delta
- Calcular el valor máximo de watermark de los datos entrantes
- Aplicar filtrado de período de retroceso (ej., "3d" = retroceso de 3 días)
- Validar la calidad de los datos antes de la escritura

In [0]:
# ==============================================================================
# SECCIÓN 8.1: Convertir a DataFrame de Spark
# ==============================================================================

try:
    print(f"Convirtiendo a DataFrame de Spark...")
    df_spark = spark.createDataFrame(df_final)
    print(f"Conversión exitosa: {df_spark.count()} filas")
    
    # Mostrar esquema para validación
    print(f"\nEsquema de Spark:")
    
    df_spark.printSchema()
    
except Exception as e:
    raise RuntimeError(f"Error en la conversión a DataFrame de Spark: {str(e)}")

# ==============================================================================
# SECCIÓN 8.2: Establecer y Calcular el Valor de Watermark de Retroceso
# ==============================================================================
current_watermark_val = watermark_val

if watermark_look_back.strip() and current_watermark_val.strip():
    current_watermark_val = adjust_value(current_watermark_val, watermark_look_back.strip())
    print(f"Retroceso ajustado:  {watermark_look_back.strip()}")
    print(f"  - Nuevo Watermark:  {current_watermark_val}")

# ==============================================================================
# SECCIÓN 8.3: Aplicar Filtrado de Watermark
# ==============================================================================

if current_watermark_val:
    try:
        print(f"Aplicando filtro de watermark: {str(current_watermark_val).strip()}")
        original_count = df_spark.count()
        
        # Filtrar solo para incluir registros después de la fecha de retroceso
        df_spark = df_spark.filter(col(watermark_column) >= lit(current_watermark_val))
        
        filtered_count = df_spark.count()
        removed_count = original_count - filtered_count
        
        print(f"   Filas antes: {original_count}")
        print(f"   Filas después:  {filtered_count}")
        print(f"   Eliminadas:     {removed_count} filas fuera de la ventana de retroceso")
        
    except Exception as e:
        print(f"Advertencia en el filtrado de retroceso: {str(e)}")

# ==============================================================================
# SECCIÓN 8.4: Validar Conjunto de Datos Final
# ==============================================================================

final_count = df_spark.count()
if final_count == 0:
    raise ValueError(
        "El conjunto de datos final está vacío después del procesamiento. "
        "Verifique los datos de entrada, los filtros y la configuración de retroceso."
    )

print(f"\nConjunto de datos final listo: {final_count} filas")

# Mostrar datos de muestra para validación manual
print(f"\nDatos de Muestra (primeras 5 filas):")
display(df_spark.limit(5))

## Sección 9: Construcción del Nombre de la Tabla y Preparación del Esquema

**Propósito**: Normalizar nombres de tabla y columna, preparar el identificador de la tabla de destino
y manejar la evolución del esquema (agregar nuevas columnas a tablas existentes).

In [0]:
# ==============================================================================
# SECCIÓN 9.1: Normalizar Nombre de Hoja de Excel
# ==============================================================================

# Estandarizar nombre de hoja para el nombrado de tablas: minúsculas, reemplazar caracteres especiales con guiones bajos
final_excel_sheet = (
    excel_sheet_name
    .lower()
    .replace(' ', '_')
    .replace('.', '')
    .replace('/', '_')
    .replace('-', '_')
)

# ==============================================================================
# SECCIÓN 9.2: Construir Nombre de la Tabla de Destino
# ==============================================================================

# Normalizar el nombre de la tabla de destino usando la utilidad slugify
table_base = (
    table_name
    .lower()
    .replace(' ', '_')
    .replace('.', '')
    .replace('/', '_')
)
final_destination_table = slugify(table_base)

# Combinar con el sufijo de pronóstico
new_table_name = f"{final_destination_table}{final_forecast_name}"

# Identificador completo de la tabla con catálogo y esquema
full_table_name = f"{environment}_bronze_landing_minsur.marcobre_fileserver.{new_table_name}"

print(f"Configuración de la Tabla de Destino:")
print(f"   Nombre base:        {table_name}")
print(f"   Estandarizado:      {final_destination_table}")
print(f"   Sufijo de pronóstico: {final_forecast_name or '(ninguno)'}")
print(f"   Nombre final:       {new_table_name}")
print(f"   Ruta completa:      {full_table_name}")

# ==============================================================================
# SECCIÓN 9.3: Manejar Evolución del Esquema (Agregar Nuevas Columnas)
# ==============================================================================

table_exists = spark.catalog.tableExists(full_table_name)

if table_exists:
    print(f"\nLa tabla existe - verificando cambios de esquema...")
    
    try:
        # Referenciar la tabla Delta existente
        existing_table = DeltaTable.forName(spark, full_table_name)
        existing_df = existing_table.toDF()
        
        # Comparar columnas
        existing_cols = set(existing_df.columns)
        new_cols = set(df_spark.columns)
        
        cols_only_in_new = new_cols - existing_cols
        cols_only_in_existing = existing_cols - new_cols
        
        print(f"   Columnas existentes: {len(existing_cols)}")
        print(f"   Nuevas columnas:      {len(new_cols)}")
        
        if cols_only_in_new:
            print(f"   Nuevas columnas a agregar: {cols_only_in_new}")
            
            try:
                # Construir declaración ALTER TABLE ADD COLUMNS
                # Obtener tipos de datos del esquema del nuevo DataFrame
                new_fields = []
                for field in df_spark.schema.fields:
                    if field.name in cols_only_in_new:
                        type_str = field.dataType.simpleString()
                        new_fields.append(f"{field.name} {type_str}")
                
                if new_fields:
                    alter_sql = f"""
                        ALTER TABLE {full_table_name}
                        ADD COLUMNS ({', '.join(new_fields)})
                    """
                    
                    print(f"   Ejecutando: {alter_sql[:100]}...")
                    spark.sql(alter_sql)
                    print(f"   Se agregaron {len(new_fields)} nueva(s) columna(s)")
                    
            except AnalysisException as e:
                if "already exists" in str(e):
                    print(f"   Las columnas pueden ya existir (modificación concurrente)")
                else:
                    print(f"   Advertencia en la actualización de esquema: {str(e)}")
        else:
            print(f"   No se detectaron cambios de esquema")
            
        if cols_only_in_existing:
            print(f"   Columnas solo en la tabla existente (serán nulas en los nuevos datos): {cols_only_in_existing}")
            
    except Exception as e:
        print(f"   Advertencia en la verificación de evolución de esquema: {str(e)}")
else:
    print(f"\nLa tabla no existe - se creará una nueva tabla")

## Sección 10: Operaciones de Tabla Delta (Escritura/Fusión)

**Propósito**: Ejecutar la operación de escritura final según el tipo de carga:
- **Carga Completa**: Modo sobrescritura (sin watermark configurado, tabla nueva)
- **Carga Incremental**: Modo Fusión/Upsert (watermark configurado, tabla existe)

**Lógica de Fusión**:
- Coincidir en columnas ID (combinación AND)
- Actualizar: Todas las columnas no ID
- Insertar: Todas las columnas para nuevos registros

In [0]:
# ==============================================================================
# SECCIÓN 10.1: Determinar Estrategia de Carga
# ==============================================================================
current_watermark_val = str(current_watermark_val)
is_full_load = (
    (not current_watermark_val or (isinstance(current_watermark_val, str) and not current_watermark_val.strip())) and 
    not table_exists
)

is_incremental_load = not is_full_load

print(f"Determinación de Estrategia de Carga:")
print(f"   Watermark proporcionado:  {bool(current_watermark_val and current_watermark_val.strip())}")
print(f"   Tabla existe:             {table_exists}")
print(f"   Estrategia:               {'CARGA COMPLETA' if is_full_load else 'CARGA INCREMENTAL'}")

# ==============================================================================
# SECCIÓN 10.2: Ejecutar Carga Completa (Sobrescritura)
# ==============================================================================

if is_full_load:
    print(f"\nEjecutando CARGA COMPLETA (modo sobrescritura)...")
    
    try:
        # Escribir con fusión de esquema para tablas nuevas
        df_spark.write \
            .format("delta") \
            .option("mergeSchema", "true") \
            .option("overwriteSchema", "true") \
            .mode("overwrite") \
            .saveAsTable(full_table_name)
        
        print(f"Carga completa completada exitosamente")
        print(f"   Filas escritas: {df_spark.count()}")
        print(f"   Destino: {full_table_name}")
        
    except Exception as e:
        raise RuntimeError(f"Error en la carga completa: {str(e)}")

# ==============================================================================
# SECCIÓN 10.3: Ejecutar Carga Incremental (Fusión/Upsert)
# ==============================================================================

else:
    print(f"\nEjecutando CARGA INCREMENTAL (modo fusión)...")
    
    # Validar que se especifiquen las columnas ID para la clave de fusión
    if not id_columns or not id_columns.strip():
        raise ValueError(
            "Se deben especificar las columnas ID para la carga incremental (operación de fusión). "
            "Configure el widget 'IdColumns' con nombres de columna separados por comas."
        )
    
    # Analizar y validar columnas ID
    id_cols_list = [col.strip() for col in id_columns.split(",") if col.strip()]
    if not id_cols_list:
        raise ValueError("El widget IdColumns está vacío o contiene solo delimitadores")
    
    # Validar que las columnas ID existan en los datos de origen
    missing_id_cols = [c for c in id_cols_list if c not in df_spark.columns]
    if missing_id_cols:
        raise ValueError(
            f"Columnas ID no encontradas en los datos de origen: {missing_id_cols}. "
            f"Columnas disponibles: {df_spark.columns}"
        )
    
    print(f"\nConfiguración de Carga Incremental:")
    print(f"   Columnas ID:        {id_cols_list}")
    print(f"   Columna Watermark:  {watermark_column}")
    print(f"   Período de Retroceso:   {watermark_look_back or '(ninguno)'}")
    print(f"   Watermark Anterior: {current_watermark_val or '(carga inicial)'}")
    
    # Referenciar la tabla Delta de destino
    try:
        target_table = DeltaTable.forName(spark, full_table_name)
    except AnalysisException as e:
        raise RuntimeError(f"Error al referenciar la tabla de destino {full_table_name}: {str(e)}")
    
    # Verificar si la tabla de destino tiene datos
    target_count = target_table.toDF().count()
    print(f"   Filas en tabla destino: {target_count}")
    
    if target_count == 0:
        print(f"\nLa tabla de destino está vacía - cambiando a modo SOLO INSERCIÓN")
        # Si el destino está vacío, podemos hacer solo inserción por rendimiento
        merge_mode = "solo_insercion"
    else:
        merge_mode = "fusion_completa"
        print(f"\nEjecutando FUSIÓN COMPLETA (upsert)...")
    
    # Preparar configuración de fusión
    # Mapa de inserción: todas las columnas del origen
    insert_merge_map = {
        col_name: f"source.{col_name}" 
        for col_name in df_spark.columns
    }
    
    # Mapa de actualización: todas las columnas excepto las ID (preservar estabilidad de la clave)
    update_merge_map = {
        col_name: f"source.{col_name}"
        for col_name in df_spark.columns
        if col_name not in id_cols_list
    }
    
    # Construir condición de fusión (combinación AND de columnas ID)
    merge_conditions = [
        f"target.{col_name} = source.{col_name}"
        for col_name in id_cols_list
    ]
    merge_condition = " AND ".join(merge_conditions)
    
    print(f"\nDetalles de Fusión:")
    print(f"   Condición:     {merge_condition}")
    #print(f"   Columnas a actualizar:   {list(update_merge_map.keys())}")
    #print(f"   Columnas a insertar:   {list(insert_merge_map.keys())}")
    print(f"   Filas origen:   {df_spark.count()}")
    
    # Ejecutar operación de fusión
    try:
        merge_builder = (
            target_table
            .alias("target")
            .merge(
                source=df_spark.alias("source"),
                condition=merge_condition
            )
        )
        
        if merge_mode == "fusion_completa":
            # Agregar cláusulas de actualización e inserción
            merge_builder = (
                merge_builder
                .whenMatchedUpdate(set=update_merge_map)
                .whenNotMatchedInsert(values=insert_merge_map)
            )
        else:
            # Modo solo inserción
            merge_builder = merge_builder.whenNotMatchedInsert(values=insert_merge_map)
        
        # Ejecutar la fusión
        merge_builder.execute()
        
        # Obtener métricas si están disponibles (Delta Lake 0.7+)
        # Nota: El método metrics() puede no estar disponible en todas las versiones
        try:
            metrics = merge_builder.metrics()
            print(f"\nMétricas de Fusión:")
            print(f"   Filas actualizadas:  {metrics.get('numUpdatedRows', 'N/A')}")
            print(f"   Filas insertadas: {metrics.get('numInsertedRows', 'N/A')}")
            print(f"   Filas eliminadas:  {metrics.get('numDeletedRows', 'N/A')}")
        except:
            print(f"\nFusión completada (métricas no disponibles)")
            
        print(f"\nCarga incremental completada exitosamente")
        
    except Exception as e:
        raise RuntimeError(f"Error en la operación de fusión: {str(e)}")

## Sección 11: Persistencia de Watermark

**Propósito**: Actualizar el valor de watermark en la base de datos de configuración para habilitar
futuras cargas incrementales. Utiliza conexión JDBC a SQL Server.

**Seguridad**: 
- Solo actualizar si el cálculo del watermark fue exitoso
- Usar consultas parametrizadas/enlazadas si es posible
- Manejar fallos de conexión de manera elegante

In [0]:
max_incremental_value = None

if watermark_column.strip():
    try:
        # Validar que la columna de watermark exista en el dataframe
        if watermark_column not in df_spark.columns:
            raise ValueError(
                f"La columna de watermark '{watermark_column}' no se encuentra en los datos. "
                f"Columnas disponibles: {df_spark.columns}"
            )
        
        # Calcular el valor máximo para el seguimiento del watermark
        max_result = df_spark.select(spark_max(col(watermark_column)).alias("max_val")).first()
        max_incremental_value = max_result["max_val"] if max_result else None
        
        if max_incremental_value:
            print(f"Valor máximo de watermark calculado: {max_incremental_value}")
        else:
            print(f"No se encontró valor de watermark (la columna puede estar vacía)")
            
    except Exception as e:
        print(f"Advertencia en el cálculo del watermark: {str(e)}")
        max_incremental_value = None

# ==============================================================================
# SECCIÓN 11.1: Validar Watermark Antes de la Actualización
# ==============================================================================

if not max_incremental_value:
    print(f"No se calculó el valor de watermark - omitiendo actualización de watermark")
    print(f"   Esto puede ocurrir si:")
    print(f"   - La columna de watermark está vacía en los datos de origen")
    print(f"   - La columna de watermark no está configurada")
    print(f"   - Todos los datos fueron filtrados por el retroceso")
else:
    print(f"Preparando actualización de watermark...")
    print(f"   ID de Registro:       {id_record}")
    print(f"   Watermark máximo:   {max_incremental_value}")
    
    # ==============================================================================
    # SECCIÓN 11.2: Obtener Conexión JDBC
    # ==============================================================================
    
    try:
        secret_scope = f"kv-velman-{environment}"
        secret_key = "velkpi-sqlconnectionstr-jdbc"
        
        print(f"   Obteniendo secreto JDBC: scope={secret_scope}, key={secret_key}")
        full_jdbc_url = dbutils.secrets.get(scope=secret_scope, key=secret_key)
        
        if not full_jdbc_url:
            raise ValueError("La cadena de conexión JDBC está vacía")
            
        print(f"   Cadena de conexión JDBC obtenida")
        
    except Exception as e:
        raise RuntimeError(f"Error al obtener el secreto de conexión JDBC: {str(e)}")
    
    # ==============================================================================
    # SECCIÓN 11.3: Ejecutar SQL de Actualización de Watermark
    # ==============================================================================
    
    # Nota: Usando formato de cadena para SQL - en producción, usar consultas parametrizadas
    # si la utilidad jdbc_connection admite declaraciones preparadas
    sql_update = f"""
        UPDATE [velkpi].[ExcelConfiguration]
        SET WatermarkVal = '{max_incremental_value}'
        WHERE Id = '{id_record}'
    """
    
    print(f"   Ejecutando actualización SQL...")
    
    try:
        with jdbc_connection(full_jdbc_url) as conn:
            statement = conn.createStatement()
            try:
                statement.execute(sql_update)
                
                # Verificar que la actualización afectó filas
                # Nota: Esto depende de las capacidades del controlador JDBC
                print(f"   Watermark actualizado exitosamente")
                print(f"      ID de Registro: {id_record}")
                print(f"      Nuevo valor: {max_incremental_value}")
                
            finally:
                statement.close()
                
    except Exception as e:
        # Registrar el error pero no fallar el trabajo - los datos ya están cargados
        print(f"   Error en la actualización de watermark: {str(e)}")
        print(f"      IMPORTANTE: ¡Puede ser necesaria una actualización manual del watermark!")
        print(f"      Ejecutar este SQL manualmente:")
        print(f"      {sql_update}")

## Sección 12: Verificación de Datos y Resumen Final

**Propósito**: Mostrar registros recientes de la tabla Delta de destino para verificar
la ingesta exitosa y proporcionar un resumen de ejecución.

**Consulta de Verificación**: 
- Ordena por columna de watermark descendente para mostrar primero los datos más recientes
- Limita a registros recientes por rendimiento
- Valida la accesibilidad de los datos

In [0]:
# ==============================================================================
# SECCIÓN 12.1: Preparar Consulta de Verificación
# ==============================================================================

# Determinar columna de ordenamiento
order_column = watermark_column if (watermark_column and watermark_column.strip()) else "1"

verification_query = f"""
SELECT * 
FROM {full_table_name}
ORDER BY {order_column} DESC
LIMIT 100
"""

print(f"Consulta de Verificación de Datos:")
print(f"   {verification_query.strip()}")

# ==============================================================================
# SECCIÓN 12.2: Ejecutar Consulta de Verificación
# ==============================================================================

try:
    verification_df = spark.sql(verification_query)
    verification_count = verification_df.count()
    
    print(f"\nResultados de Verificación:")
    print(f"   Filas devueltas: {verification_count}")
    
    if verification_count > 0:
        print(f"\nRegistros Más Recientes (ordenados por {order_column}):")
        display(verification_df.limit(10))
    else:
        print(f"   No se devolvieron datos de la consulta de verificación")
        
except Exception as e:
    print(f"   Error en la consulta de verificación: {str(e)}")

# ==============================================================================
# SECCIÓN 12.3: Resumen de Ejecución
# ==============================================================================

print(f"\n{'='*80}")
print(f"INGESTA INCREMENTAL CON DELTA COMPLETADA")
print(f"{'='*80}")
print(f"Resumen:")
print(f"   ID de Registro:        {id_record}")
print(f"   Archivo Origen:      {input_excel_path}")
print(f"   Tabla Destino:     {full_table_name}")
print(f"   Filas Procesadas:   {final_count}")
print(f"   Tipo de Carga:        {'CARGA COMPLETA' if is_full_load else 'CARGA INCREMENTAL'}")
print(f"   Valor de Watermark:  {max_incremental_value or 'N/A'}")
print(f"   Cambios de Esquema:   {'Sí' if (table_exists and cols_only_in_new) else 'No'}")
print(f"{'='*80}")